# FA Zone Mask Debug Notebook

Use this notebook to interactively inspect thresholding, circle detection, and final zone masks on a few annotated FA images.

In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from preprocessing.extract_fa_zone_masks import (
    Geometry,
    build_label_map,
    build_retina_mask,
    build_zone_masks,
    choose_concentric_pair,
    detect_axes,
    detect_disc_circle,
    detect_yellow_overlay,
    geometry_to_json,
    hough_circle_candidates,
    load_rgb,
    make_qc_overlay,
    orient_axes,
    refine_radii,
)

plt.rcParams["figure.figsize"] = (16, 10)
plt.rcParams["image.cmap"] = "gray"

## 1. Point this notebook at your annotated FA images

In [ ]:
INPUT_GLOB = "dataset/**/*.png"

MIN_RETINA_THRESHOLD = 8
YELLOW_S_THRESHOLD = 30
YELLOW_V_THRESHOLD = 80

In [ ]:
image_paths = sorted(p for p in REPO_ROOT.glob(INPUT_GLOB) if p.is_file())
print(f"Matched {len(image_paths)} image(s)")
for path in image_paths[:10]:
    print(path)

## 2. Helpers

In [ ]:
def show_threshold_debug(image_path, retina_threshold=MIN_RETINA_THRESHOLD, yellow_s_threshold=YELLOW_S_THRESHOLD, yellow_v_threshold=YELLOW_V_THRESHOLD):
    rgb = load_rgb(Path(image_path))
    retina_mask = build_retina_mask(rgb, threshold=retina_threshold)
    yellow_mask = detect_yellow_overlay(rgb, sat_threshold=yellow_s_threshold, val_threshold=yellow_v_threshold)

    circles = None
    hough_error = None
    try:
        circles = hough_circle_candidates(yellow_mask)
    except Exception as exc:
        hough_error = str(exc)

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    axes[0].imshow(rgb)
    axes[0].set_title("Annotated FA")
    axes[1].imshow(yellow_mask, cmap="gray")
    axes[1].set_title("Yellow Overlay Mask")
    axes[2].imshow(retina_mask, cmap="gray")
    axes[2].set_title("Retina Mask")
    axes[3].imshow(rgb)
    axes[3].set_title("Hough Circle Candidates")

    if circles is not None:
        for x, y, r in circles:
            axes[3].add_patch(patches.Circle((x, y), r, fill=False, edgecolor="cyan", linewidth=2))
            axes[3].plot(x, y, marker="+", color="red", markersize=10)
    else:
        axes[3].text(0.02, 0.98, f"No circles found\n{hough_error}", transform=axes[3].transAxes, va="top", color="white", bbox=dict(facecolor="black", alpha=0.7))

    for ax in axes:
        ax.axis("off")
    plt.suptitle(str(image_path), fontsize=14)
    plt.tight_layout()
    plt.show()

    return {
        "rgb": rgb,
        "retina_mask": retina_mask,
        "yellow_mask": yellow_mask,
        "hough_circles": circles,
        "hough_error": hough_error,
    }


def extract_debug_artifacts(image_path, retina_threshold=MIN_RETINA_THRESHOLD, yellow_s_threshold=YELLOW_S_THRESHOLD, yellow_v_threshold=YELLOW_V_THRESHOLD):
    rgb = load_rgb(Path(image_path))
    retina_mask = build_retina_mask(rgb, threshold=retina_threshold)
    yellow_mask = detect_yellow_overlay(rgb, sat_threshold=yellow_s_threshold, val_threshold=yellow_v_threshold)

    result = {
        "rgb": rgb,
        "retina_mask": retina_mask,
        "yellow_mask": yellow_mask,
        "hough_circles": None,
    }

    circles = hough_circle_candidates(yellow_mask)
    result["hough_circles"] = circles
    outer_circle, inner_circle = choose_concentric_pair(circles, rgb.shape[:2])
    center_xy = (
        float(outer_circle[0] + inner_circle[0]) / 2.0,
        float(outer_circle[1] + inner_circle[1]) / 2.0,
    )
    inner_radius, outer_radius = refine_radii(
        yellow_mask,
        center_xy=center_xy,
        outer_radius_guess=float(outer_circle[2]),
        inner_radius_guess=float(inner_circle[2]),
    )
    disc_center_xy, disc_radius = detect_disc_circle(
        yellow_mask,
        center_xy=center_xy,
        inner_radius=inner_radius,
        outer_radius=outer_radius,
    )
    axis_a, axis_b = detect_axes(
        yellow_mask,
        center_xy=center_xy,
        inner_radius=inner_radius,
        outer_radius=outer_radius,
    )
    disc_axis, vertical_axis, eye = orient_axes(disc_center_xy, center_xy, axis_a, axis_b)

    geometry = Geometry(
        center_xy=center_xy,
        inner_radius=inner_radius,
        outer_radius=outer_radius,
        disc_center_xy=disc_center_xy,
        disc_radius=disc_radius,
        eye=eye,
        disc_axis_xy=(float(disc_axis[0]), float(disc_axis[1])),
        vertical_axis_xy=(float(vertical_axis[0]), float(vertical_axis[1])),
    )
    zone_masks = build_zone_masks(rgb.shape[:2], retina_mask, geometry)
    label_map = build_label_map(zone_masks)
    qc_overlay = make_qc_overlay(rgb, zone_masks, geometry)

    result.update({
        "geometry": geometry,
        "geometry_json": geometry_to_json(geometry),
        "zone_masks": zone_masks,
        "label_map": label_map,
        "qc_overlay": qc_overlay,
    })
    return result


def show_debug_result(image_path, retina_threshold=MIN_RETINA_THRESHOLD, yellow_s_threshold=YELLOW_S_THRESHOLD, yellow_v_threshold=YELLOW_V_THRESHOLD):
    threshold_debug = show_threshold_debug(
        image_path,
        retina_threshold=retina_threshold,
        yellow_s_threshold=yellow_s_threshold,
        yellow_v_threshold=yellow_v_threshold,
    )

    try:
        result = extract_debug_artifacts(
            image_path,
            retina_threshold=retina_threshold,
            yellow_s_threshold=yellow_s_threshold,
            yellow_v_threshold=yellow_v_threshold,
        )
    except Exception as exc:
        print(f"Extraction failed: {exc}")
        print("Tune the yellow thresholds until the circle candidates match the overlaid rings.")
        return threshold_debug

    fig, axes = plt.subplots(4, 4, figsize=(18, 16))
    axes = axes.ravel()

    panels = [
        ("Annotated FA", result["rgb"]),
        ("Yellow Overlay Mask", result["yellow_mask"]),
        ("Retina Mask", result["retina_mask"]),
        ("QC Overlay", result["qc_overlay"]),
    ]
    for zone in range(1, 11):
        panels.append((f"Zone {zone}", result["zone_masks"][zone]))

    for ax, (title, panel) in zip(axes, panels):
        if panel.ndim == 2:
            ax.imshow(panel, cmap="gray")
        else:
            ax.imshow(panel)
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[len(panels):]:
        ax.axis("off")

    plt.suptitle(str(image_path), fontsize=14)
    plt.tight_layout()
    plt.show()

    print(json.dumps(result["geometry_json"], indent=2))
    return result

## 3. Test one image

In [ ]:
IMAGE_INDEX = 0

if not image_paths:
    raise ValueError("No images matched INPUT_GLOB. Update the glob above and rerun.")

result = show_debug_result(
    image_paths[IMAGE_INDEX],
    retina_threshold=MIN_RETINA_THRESHOLD,
    yellow_s_threshold=YELLOW_S_THRESHOLD,
    yellow_v_threshold=YELLOW_V_THRESHOLD,
)

## 4. Optional sliders

In [ ]:
try:
    import ipywidgets as widgets

    if not image_paths:
        raise ValueError("No images matched INPUT_GLOB. Update the glob above and rerun.")

    @widgets.interact(
        image_index=widgets.IntSlider(value=0, min=0, max=max(len(image_paths) - 1, 0), step=1),
        retina_threshold=widgets.IntSlider(value=MIN_RETINA_THRESHOLD, min=0, max=40, step=1),
        yellow_s_threshold=widgets.IntSlider(value=YELLOW_S_THRESHOLD, min=0, max=255, step=5),
        yellow_v_threshold=widgets.IntSlider(value=YELLOW_V_THRESHOLD, min=0, max=255, step=5),
    )
    def _interactive_debug(image_index, retina_threshold, yellow_s_threshold, yellow_v_threshold):
        show_debug_result(
            image_paths[image_index],
            retina_threshold=retina_threshold,
            yellow_s_threshold=yellow_s_threshold,
            yellow_v_threshold=yellow_v_threshold,
        )

except ModuleNotFoundError:
    print("ipywidgets is not installed. Use the manual cell above, or install ipywidgets for sliders.")

## 5. Save outputs for the current image

In [ ]:
if "zone_masks" not in result:
    raise ValueError("The current result only contains threshold-debug output. Tune thresholds until extraction succeeds, then rerun this save cell.")

OUTPUT_DIR = REPO_ROOT / "notebook_debug_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

selected_image = image_paths[IMAGE_INDEX]
selected_output_dir = OUTPUT_DIR / Path(selected_image).stem
selected_output_dir.mkdir(exist_ok=True)

Image.fromarray(result["qc_overlay"]).save(selected_output_dir / "qc_overlay.png")
Image.fromarray(result["label_map"], mode="L").save(selected_output_dir / "label_map.png")
Image.fromarray((result["retina_mask"].astype(np.uint8) * 255), mode="L").save(selected_output_dir / "retina_mask.png")
Image.fromarray((result["yellow_mask"].astype(np.uint8) * 255), mode="L").save(selected_output_dir / "yellow_overlay_mask.png")

for zone in range(1, 11):
    Image.fromarray((result["zone_masks"][zone].astype(np.uint8) * 255), mode="L").save(selected_output_dir / f"zone_{zone:02d}.png")

with open(selected_output_dir / "geometry.json", "w", encoding="utf-8") as f:
    json.dump(result["geometry_json"], f, indent=2)

print(f"Saved outputs to {selected_output_dir}")